In [2]:
%matplotlib inline
import torch
import torchvision
from torch.utils import data
from torchvision import transforms
from IPython import display


In [3]:
import IPython
print(IPython.__version__) # 作为jupyter的依赖自动安装好了

9.15.0


In [4]:
def get_dataloader_workers():
    """使用4个进程读取数据"""
    return 0

In [5]:
# 整合所有的组件, 获取FashionMNIST数据集，返回训练集和验证集的数据迭代器

def load_data_fashion_mnist(batch_size, resize=None):
    """下载FashionMNIST的数据集，然后将其加载到内存中"""
    trans = [transforms.ToTensor()]
    if resize:
        trans.insert(0, transforms.Resize(resize))
    trans = transforms.Compose(trans)
    mnist_train = torchvision.datasets.FashionMNIST(
        root="../data", train=True, transform=trans, download=True
    ) 
    mnist_test = torchvision.datasets.FashionMNIST(
        root="../data", train=False, transform=trans, download=True
    ) 

    return (data.DataLoader(mnist_train, batch_size, shuffle=True, 
                num_workers=get_dataloader_workers()),
            data.DataLoader(mnist_train, batch_size, shuffle=True,
                num_workers=get_dataloader_workers()))

In [6]:
batch_size = 32
train_iter, test_iter = load_data_fashion_mnist(batch_size, resize=None)

In [7]:
num_inputs = 784 # 28 * 28
num_outputs = 10

In [8]:
display.display("hello")

'hello'

In [9]:
W = torch.normal(0, 0.01, size=(num_inputs, num_outputs), requires_grad=True)
b = torch.zeros(num_outputs, requires_grad=True)
W, W.shape, b, b.shape

(tensor([[ 0.0124,  0.0050,  0.0172,  ..., -0.0029, -0.0118,  0.0120],
         [-0.0154, -0.0115,  0.0073,  ..., -0.0058,  0.0096, -0.0041],
         [-0.0237,  0.0028, -0.0006,  ...,  0.0014, -0.0076, -0.0168],
         ...,
         [-0.0199,  0.0032,  0.0201,  ...,  0.0069,  0.0096,  0.0031],
         [ 0.0037,  0.0061, -0.0004,  ..., -0.0081, -0.0064,  0.0081],
         [ 0.0034,  0.0038, -0.0097,  ...,  0.0130, -0.0095, -0.0114]],
        requires_grad=True),
 torch.Size([784, 10]),
 tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True),
 torch.Size([10]))

In [10]:
# 定义softmax操作

X = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
X.sum(0, keepdim=True), X.sum(1, keepdim=True)

(tensor([[5., 7., 9.]]),
 tensor([[ 6.],
         [15.]]))

In [11]:
def softmax(X):
    X_exp = torch.exp(X)
    partition = X_exp.sum(1, keepdim=True)
    return X_exp / partition # 这里使用了广播机制

In [31]:
X = torch.normal(0, 1, (2, 5))
print(X)
print()
X_prob = softmax(X)
print(X_prob, X_prob.sum(1), sep='\n\n')

tensor([[-2.0628, -0.3067,  0.4461, -0.2595, -2.3331],
        [-0.6140, -0.1024,  0.6392,  0.7005,  1.6183]])

tensor([[0.0386, 0.2234, 0.4743, 0.2342, 0.0295],
        [0.0520, 0.0868, 0.1822, 0.1938, 0.4851]])

tensor([1.0000, 1.0000])


In [13]:
def net(X):
    return softmax(torch.matmul(X.reshape(-1, W.shape[0]), W) + b)

In [ ]:
# 定义损失函数
y = torch.tensor([0, 2])
print(y)
print()
y_hat = torch.tensor([[0.1, 0.3, 0.6],[0.3, 0.2, 0.5]])
print(y_hat[[0, 1], y])
print()
print(y_hat[:, y]) # 这俩还有区别
# x[[0,2], [1,0]] 配对取 (0,1)、(2,0)

tensor([0, 2])

tensor([0.1000, 0.5000])

tensor([[0.1000, 0.6000],
        [0.3000, 0.5000]])


In [15]:
def cross_entropy(y_hat, y):
    return -torch.log(y_hat[range(0, len(y_hat)), y])

cross_entropy(y_hat, y)

tensor([2.3026, 0.6931])

In [16]:
def accuracy(y_hat, y):
    """计算预测正确的数量"""
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1: 
        # 确保第一轴长度大于1，第二轴长度大于1
        y_hat = y_hat.argmax(axis = 1) # 得到每次预测的最大的概率的值的索引
    cmp = y_hat.type(y.dtype) == y
    # print(cmp)
    return float(cmp.type(y.dtype).sum())

In [17]:
accuracy(y_hat, y) / len(y)

0.5

In [18]:
class Accumulator:
    """在n个变量上累加"""
    def __init__(self, n):
        self.data = [0.0] * n

    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]

    def reset(self):
        self.data = [0.0] * len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [19]:
# 评估任意模型的精度

def evaluate_accuracy(net, data_iter):
    """计算指定数据集上模型的精度"""
    if isinstance(net, torch.nn.Module):
        net.eval() # 将模型设置为评估模式
    metric = Accumulator(2) #正确预测数，预测总数
    with torch.no_grad():
        for X, y in data_iter:
            metric.add(accuracy(net(X), y), y.numel())

    return metric[0] / metric[1]

In [20]:
evaluate_accuracy(net, test_iter)

0.05391666666666667

In [21]:
# 训练
def train_epoch_ch3(net, train_iter, loss, updater):
    """训练模型一轮"""
    # 将模型设置为训练模式
    if isinstance(net, torch.nn.Module):
        net.train()
    